# 00 · Raw-data inventory

Step 1 of understanding the data: what is on disk, how big it is, which files the pipeline actually uses, what is still missing, and whether every study tract is present in every source.

In [1]:
import warnings, os, json, glob
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name in ("eda", "notebooks") else Path.cwd()
os.chdir(ROOT)
import numpy as np, pandas as pd, matplotlib.pyplot as plt, duckdb, pyarrow.parquet as pq
pd.set_option("display.width", 170); pd.set_option("display.max_columns", 30); pd.set_option("display.precision", 3)
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": .25})
REG = ["maricopa-az", "northern-ca", "eastern-ok", "south-central-tx", "eastern-wa"]
COL = dict(zip(REG, ["#e07b39", "#c0504d", "#6a9e4f", "#3b6fb6", "#8e6bbf"]))

def challenge_root():
    """data/raw/challenge (Windows junctions made by fetch_data.ps1), or the same files in ../MapEquityChallenge/data
    where the junction cannot be followed (e.g. from Linux/WSL)."""
    p = Path("data/raw/challenge")
    try:
        if (p / "reference" / "maricopa-az").is_dir():
            return p
    except OSError:
        pass
    return Path("../MapEquityChallenge/data")
CH = challenge_root(); RAW = Path("data/raw")
TAB, FIG = Path("eda/tables"), Path("eda/figures"); TAB.mkdir(parents=True, exist_ok=True); FIG.mkdir(parents=True, exist_ok=True)
def save(name):
    plt.tight_layout(); plt.savefig(FIG / f"{name}.png", dpi=130, bbox_inches="tight"); plt.show()
con = duckdb.connect(); con.sql("SET threads=4")
study = pd.concat([pd.read_csv(CH / f"reference/{r}/{r}-sample-submission.csv", dtype={"GEOID": str}, usecols=["GEOID"]).assign(region=r) for r in REG])
print("challenge data read from:", CH.resolve(), "| study tracts:", len(study))

challenge data read from: /sessions/rcw-013hmcfk1naa7pch2jpe8lbk/mnt/MapEquityChallenge/data | study tracts: 9794


## 1. Challenge extracts (Overture 2026-08-19.0) by region

In [2]:
USED = {"overture-buildings.parquet": "yes (buildings)", "overture-pois.parquet": "yes (places)", "sample-submission.csv": "yes (tract list)"}
rows = []
for r in REG:
    for f in sorted((CH / "reference" / r).iterdir()):
        layer = f.name.replace(r + "-", "")
        n = pq.ParquetFile(f).metadata.num_rows if f.suffix == ".parquet" else sum(1 for _ in open(f)) - 1
        rows.append({"region": r, "file": layer, "MB": f.stat().st_size / 1e6, "rows": n, "used_by_pipeline": USED.get(layer, "no")})
ref = pd.DataFrame(rows); ref.to_csv(TAB / "00_challenge_reference_files.csv", index=False)
piv = ref.pivot_table(index="file", columns="region", values="rows", aggfunc="sum")[REG]
piv["MB_total"] = ref.groupby("file").MB.sum(); piv["used_by_pipeline"] = ref.groupby("file").used_by_pipeline.first()
display(piv.style.format({r: "{:,.0f}" for r in REG} | {"MB_total": "{:,.0f}"}))
print(f"total {ref.MB.sum() / 1e3:.1f} GB, of which used by the pipeline {ref[ref.used_by_pipeline != 'no'].MB.sum() / 1e3:.1f} GB")

region,maricopa-az,northern-ca,eastern-ok,south-central-tx,eastern-wa,MB_total,used_by_pipeline
file,,,,,,,
census-acs-housing.csv,"1,593",591,"1,192","6,010",415,0,no
census-acs-housing.parquet,"1,593",591,"1,192","6,010",415,26,no
overture-buildings.parquet,"2,908,224","1,164,724","2,551,694","11,463,801","1,081,346","2,149",yes (buildings)
overture-infrastructure.parquet,"9,482","3,318","2,726","13,622","4,131",3,no
overture-pois.parquet,"300,046","116,897","207,369","1,300,334","76,491",277,yes (places)
overture-rail.parquet,"3,830","2,006","5,066","17,895","3,473",12,no
overture-roads-unfiltered.parquet,"1,170,165","479,596","1,037,555","4,122,701","383,136","1,512",no
overture-roads.parquet,"494,492","191,836","520,550","1,937,009","142,649",537,no
sample-submission.csv,"1,593",591,"1,192","6,003",415,0,yes (tract list)


total 4.5 GB, of which used by the pipeline 2.4 GB


**Takeaway.** The challenge extracts are 4.5 GB, but the pipeline reads only three of the nine file types per region: buildings, places and the tract list (2.4 GB). Roads, rail, infrastructure and the challenge's own ACS housing extract are not used (MapEquityChallenge used the roads for its highway analysis). Buildings dominate the volume: 11.5 million in south-central Texas alone.

## 2. Challenge strata

In [3]:
rows = []
for f in sorted((CH / "strata").rglob("*.parquet")):
    rows.append({"folder": f.parent.name, "file": f.name, "MB": f.stat().st_size / 1e6, "rows": pq.ParquetFile(f).metadata.num_rows,
                 "columns": len(pq.ParquetFile(f).schema_arrow)})
st = pd.DataFrame(rows); st.to_csv(TAB / "00_challenge_strata_files.csv", index=False)
print(len(st), "parquet files,", f"{st.MB.sum() / 1e3:.2f} GB")
used = ["national-census-tract-table.parquet", "national-strata-tract-table.parquet"] + [f"{r}-census-tracts.parquet" for r in REG]
display(st[st.file.isin(used)].set_index("file")[["folder", "rows", "columns", "MB"]])

144 parquet files, 0.60 GB


,folder,rows,columns,MB
file,,,,
eastern-ok-census-tracts.parquet,eastern-ok,1192,14,5.325
eastern-wa-census-tracts.parquet,eastern-wa,415,14,3.872
maricopa-az-census-tracts.parquet,maricopa-az,1593,14,5.242
national-census-tract-table.parquet,national,85396,19,4.487
national-strata-tract-table.parquet,national,85396,232,38.063
northern-ca-census-tracts.parquet,northern-ca,591,14,4.762
south-central-tx-census-tracts.parquet,south-central-tx,6010,14,20.581


**Takeaway.** Of 144 strata files (0.6 GB), the pipeline uses seven: the national census-tract spine (land area and centroid for 85,396 tracts), the national strata table (232 columns, audit only) and the five regional tract-polygon files. The Texas polygon file has 6,010 tracts, but only 6,003 are in the study's tract list, so seven Texas tracts are deliberately outside the study.

## 3. Downloaded inputs (scripts/fetch_data.ps1)

In [4]:
man = pd.read_csv(RAW / "MANIFEST.csv")
man["file"] = man.file.str.replace("\\\\", "/", regex=True).str.replace("./", "", regex=False)
def nrows(p):
    p = Path(p)
    if p.suffix == ".parquet": return pq.ParquetFile(p).metadata.num_rows
    if p.suffix == ".zip": return np.nan
    if p.name.endswith(".gz"): return len(pd.read_csv(p, usecols=[0]))
    return sum(1 for _ in open(p, errors="ignore")) - 1
man["rows"] = [nrows(f) for f in man.file]; man["MB"] = man.bytes / 1e6
man.to_csv(TAB / "00_downloaded_files.csv", index=False)
display(man[["file", "MB", "rows", "fetched"]].style.format({"MB": "{:.1f}", "rows": "{:,.0f}"}))
pending = {"USA Structures counts": "data/processed/usa_structures_tract.parquet", "Overture 2026-10 counts": "data/processed",
           "Overture 2023 counts": "data/processed"}
proc = [p.name for p in Path("data/processed").glob("*.parquet")]
print("data/processed:", proc if proc else "empty -> run scripts/fetch_data.ps1 -Stream (USA Structures, Overture 2026-10 and 2023)")

,file,MB,rows,fetched
0,data/raw/lodes/az_wac_S000_JT00_2022.csv.gz,0.9,"36,998",2026-11-10T14:23:17
1,data/raw/lodes/ca_wac_S000_JT00_2022.csv.gz,6.2,"257,557",2026-11-10T14:22:44
2,data/raw/lodes/nm_wac_S000_JT00_2022.csv.gz,0.4,"17,161",2026-11-10T14:23:23
3,data/raw/lodes/ok_wac_S000_JT00_2022.csv.gz,0.8,"33,305",2026-11-10T14:23:42
4,data/raw/lodes/tx_wac_S000_JT00_2022.csv.gz,4.1,"165,213",2026-11-10T14:23:06
5,data/raw/lodes/wa_wac_S000_JT00_2022.csv.gz,1.3,"53,670",2026-11-10T14:23:31
6,data/raw/cbp/cbp23co.txt,107.7,"1,100,961",2025-05-29T16:48:42
7,data/raw/cbp/cbp23co.zip,12.9,nan,2026-11-10T14:23:12
8,data/raw/cbp/tab20_zcta520_tract20_natl.txt,24.3,"171,480",2026-11-10T14:23:37
9,data/raw/cbp/zbp23totals.txt,2.9,"34,954",2025-06-09T10:51:58


data/processed: ['cbp_county.parquet', 'overture_r2026a_buildings.parquet', 'overture_r2026a_places.parquet', 'strata_audit.parquet', 'tracts_all.parquet', 'usa_structures_tract.parquet', 'zbp_tract.parquet', '_ck_r2026a_buildings_eastern-ok.parquet', '_ck_r2026a_buildings_eastern-wa.parquet', '_ck_r2026a_buildings_maricopa-az.parquet', '_ck_r2026a_buildings_northern-ca.parquet', '_ck_r2026a_buildings_south-central-tx.parquet', '_ck_r2026a_places_eastern-ok.parquet', '_ck_r2026a_places_eastern-wa.parquet', '_ck_r2026a_places_maricopa-az.parquet', '_ck_r2026a_places_northern-ca.parquet', '_ck_r2026a_places_south-central-tx.parquet', '_ck_r2026b_buildings_eastern-ok.parquet', '_ck_r2026b_buildings_eastern-wa.parquet', '_ck_r2026b_buildings_maricopa-az.parquet', '_ck_r2026b_buildings_northern-ca.parquet']


**Takeaway.** All 17 downloaded files are present with the expected sizes: each ACS table covers all 85,382 US tracts, LODES has 563,904 job blocks across six states, County Business Patterns 3,192 counties and ZIP Business Patterns 34,954 ZIP codes. data/processed is still empty: USA Structures and the 2026-10 and 2023 Overture counts arrive only with the -Stream run.

## 4. Is every study tract in every source?

In [5]:
print(study.groupby("region").size().reindex(REG).to_dict(), "| unique:", study.GEOID.is_unique, "| 11 digits:", study.GEOID.str.fullmatch(r"\d{11}").all())
geo = pd.concat([pq.read_table(CH / f"strata/{r}/{r}-census-tracts.parquet", columns=["GEOID"]).to_pandas().assign(region=r) for r in REG])
acs = pd.read_parquet(RAW / "acs/acsdt5y2024-b25001.tracts.parquet")
spine = pd.read_parquet(CH / "strata/national/national-census-tract-table.parquet", columns=["GEOID", "ALAND", "INTPTLAT"])
strata = pd.read_parquet(CH / "strata/national/national-strata-tract-table.parquet", columns=["GEOID"])
cov = pd.DataFrame({"tract polygon": study.GEOID.isin(geo.GEOID), "census spine (land, centroid)": study.GEOID.isin(spine.GEOID),
                    "ACS 2020-2024": study.GEOID.isin(acs.GEOID), "strata table": study.GEOID.isin(strata.GEOID)})
c = cov.groupby(study.region.values).mean().reindex(REG); c.loc["all"] = cov.mean()
display(c.style.format("{:.2%}"))
print("states in the study:", sorted(study.GEOID.str[:2].unique()), "| tract polygons per region file:", geo.groupby("region").size().reindex(REG).to_dict())

{'maricopa-az': 1593, 'northern-ca': 591, 'eastern-ok': 1192, 'south-central-tx': 6003, 'eastern-wa': 415} | unique: True | 11 digits: True


,tract polygon,"census spine (land, centroid)",ACS 2020-2024,strata table
maricopa-az,100.00%,100.00%,100.00%,100.00%
northern-ca,100.00%,100.00%,100.00%,100.00%
eastern-ok,100.00%,100.00%,100.00%,100.00%
south-central-tx,100.00%,100.00%,100.00%,100.00%
eastern-wa,100.00%,100.00%,100.00%,100.00%
all,100.00%,100.00%,100.00%,100.00%


states in the study: ['04', '06', '35', '40', '48', '53'] | tract polygons per region file: {'maricopa-az': 1593, 'northern-ca': 591, 'eastern-ok': 1192, 'south-central-tx': 6010, 'eastern-wa': 415}


**Takeaway.** All 9,794 study tracts are unique, keep their 11-digit IDs, and appear in every source the pipeline joins: tract polygons, census spine, ACS and the strata table (100% each), so no join silently drops tracts. The study spans six states, not five: one tract (35023970000, Hidalgo County) is in New Mexico, which is why LODES is downloaded for NM.

## 5. Audit strata: coverage only (never compared with outcomes)

In [6]:
A = ["svi_overall", "cvi_overall", "ruca_primary", "tribal_any", "epht_heat_days_summer", "usfs_BP_mean", "usdm_summer_dsci"]
sa = pd.read_parquet(CH / "strata/national/national-strata-tract-table.parquet", columns=["GEOID"] + A)
sa = study.merge(sa, on="GEOID", how="left")
miss = sa[A].isna().groupby(sa.region).mean().reindex(REG).T; miss["all"] = sa[A].isna().mean()
if "svi_overall" in sa: miss.loc["svi_overall < 0 (suppressed code)"] = (sa.svi_overall < 0).groupby(sa.region).mean().reindex(REG).tolist() + [(sa.svi_overall < 0).mean()]
miss.to_csv(TAB / "00_audit_strata_missing.csv")
display(miss.style.format("{:.1%}"))

region,maricopa-az,northern-ca,eastern-ok,south-central-tx,eastern-wa,all
svi_overall,1.8%,0.0%,0.9%,0.7%,0.7%,0.9%
cvi_overall,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%
ruca_primary,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%
tribal_any,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%
epht_heat_days_summer,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%
usfs_BP_mean,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%
usdm_summer_dsci,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%
svi_overall < 0 (suppressed code),0.0%,0.0%,0.0%,0.0%,0.0%,0.0%


**Takeaway.** The audit variables are essentially complete for the study tracts: SVI is missing for 0.9% (1.8% in Maricopa); CVI, RUCA, tribal overlap, heat, burn probability and drought are complete. This is coverage only; no audit variable is compared with any outcome here.

## Summary

Everything the pipeline needs is on disk except the cloud-streamed parts (USA Structures and the two other Overture releases), every study tract is present in every source, and only about half of the challenge download is actually used. Next: open the Overture files themselves (01) and the census inputs (02).